In [1]:
%load_ext autoreload
%autoreload 2
%env ANYWIDGET_HMR=1

env: ANYWIDGET_HMR=1


In [2]:
import os
import json
import numpy as np
import pandas as pd
import celldega as dega
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import Polygon
import matplotlib.patches as mpatches

## Inputs

In [3]:
default_data_path = "data/Xenium_Prime_Human_Skin_FFPE_outs/"
custom_data_path = "data/processed_data/xenium_skin/cellpose2/"
output_path = "data/processed_data/xenium_skin/cellpose2_xenium_default_merged/"

clusters_within_cutout_region = [17] # clusters according to xenium default clustering

inv_alpha_value_for_cutout_region = 100
buffer_for_cutout_region_alpha_shape = 50

## Merge different segmentations together

In [ ]:
dega.pre.merge_segmentation(default_data_path, 
                   custom_data_path, 
                   output_path, 
                   clusters_within_cutout_region, 
                   inv_alpha_value_for_cutout_region, 
                   buffer_for_cutout_region_alpha_shape)

Completed reading cell boundary and default clustering files.


/Users/jishar/anaconda3/envs/celldega_env_latest/lib/python3.11/site-packages/geopandas/geodataframe.py:1528: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)
/Users/jishar/Documents/celldega/src/celldega/pre/merge_segmentations.py:326: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cells_custom_within_cutout_region.drop(
/Users/jishar/anaconda3/envs/celldega_env_latest/lib/python3.11/site-packages/geopandas/geodataframe.py:1528: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[

Default segmented and custom segmented cells within cutout region extracted.
Alphashapes saved.
Resolving boundary region cell conflicts...
Merged Segmentation saved.
Merged Segmentation Meta data saved.
Calculating new assignment of transcripts...


## Visualize outputs in Matplotlib (Sanity Check)

In [ ]:
largest_cutout_region_alpha_shape = gpd.read_parquet(f"{output_path}/cutout_region_alpha_shape.parquet")
fig, ax = plt.subplots(1, 1, figsize=(40, 20))
gpd.GeoSeries(largest_cutout_region_alpha_shape.loc[0]).boundary.plot(ax=ax, linewidth=2, edgecolor='red', label='Cutout Region with Buffer')
plt.gca().invert_yaxis()
plt.title("Alphashapes", fontsize=30)
plt.legend(loc='upper left', fontsize=30)
plt.xticks(fontsize=20)
plt.yticks(fontsize=20)
plt.show()
plt.close()

default_cell_boundaries = pd.read_parquet(f"{default_data_path}cell_boundaries.parquet")
grouped = default_cell_boundaries.groupby("cell_id")[["vertex_x", "vertex_y"]].agg(
    lambda x: x.tolist()
)
grouped["geometry"] = grouped.apply(
    lambda row: Polygon(zip(row["vertex_x"], row["vertex_y"])), axis=1
)
merged_cells = gpd.GeoDataFrame(grouped, geometry="geometry")[["geometry"]]

fig, ax = plt.subplots(1, 1, figsize=(40, 40))
merged_cells.plot(ax=ax, alpha=1, linewidth=1, facecolor='white', edgecolor='red')
plt.gca().invert_yaxis()
plt.title("Cell polygons - Xenium Default", fontsize=30)
plt.xticks(fontsize=20)
plt.yticks(fontsize=20)
plt.show()
plt.close()

merged_cells = gpd.read_parquet(f"{custom_data_path}cell_polygons.parquet")
fig, ax = plt.subplots(1, 1, figsize=(40, 40))
merged_cells.plot(ax=ax, alpha=1, linewidth=1, facecolor='white', edgecolor='red')
plt.gca().invert_yaxis()
plt.title("Cell polygons - Cellpose2", fontsize=30)
plt.xticks(fontsize=20)
plt.yticks(fontsize=20)
plt.show()
plt.close()

merged_cells = gpd.read_parquet(f"{output_path}/cell_polygons.parquet")
fig, ax = plt.subplots(1, 1, figsize=(40, 40))
merged_cells.plot(ax=ax, alpha=1, linewidth=1, facecolor='white', edgecolor = ['blue' if tech != 'Xenium' else 'red' for tech in merged_cells['technology'].to_list()])
red_patch = mpatches.Patch(color='red', label='Xenium', linewidth=1)
blue_patch = mpatches.Patch(color='blue', label='Cellpose2', linewidth=1)
ax.legend(handles=[red_patch, blue_patch], fontsize=12, loc='upper right')
plt.gca().invert_yaxis()
plt.title("Cell polygons after merging of cells from Cellpose2 (Dermis region) and Xenium (Rest of the Tissue)", fontsize=30)
plt.xticks(fontsize=20)
plt.yticks(fontsize=20)
plt.show()
plt.close()

newly_assigned_transcripts = pd.read_parquet(f"{output_path}/transcripts.parquet")
fig, ax = plt.subplots(figsize=(40, 40))
ax.scatter(newly_assigned_transcripts[newly_assigned_transcripts['cell_index']!='UNASSIGNED']['x_location'], 
           newly_assigned_transcripts[newly_assigned_transcripts['cell_index']!='UNASSIGNED']['y_location'], color='red', s=1)
plt.gca().invert_yaxis()
plt.title("Assigned Transcripts after Harmonization", fontsize=30)
plt.xticks(fontsize=20)
plt.yticks(fontsize=20)
plt.show()
plt.close()

fig, ax = plt.subplots(figsize=(40, 40))
ax.scatter(newly_assigned_transcripts[newly_assigned_transcripts['cell_index']=='UNASSIGNED']['x_location'], 
           newly_assigned_transcripts[newly_assigned_transcripts['cell_index']=='UNASSIGNED']['y_location'], color='red', s=1)
plt.gca().invert_yaxis()
plt.title("Unssigned Transcripts after Harmonization", fontsize=30)
plt.xticks(fontsize=20)
plt.yticks(fontsize=20)
plt.show()
plt.close()

## Visualize outputs in Celldega

In [ ]:
path_landscape_files="data/xenium_landscape_files/Xenium_Prime_Human_Skin_FFPE_outs"
path_segmentation_files="data/processed_data/xenium_skin/cellpose2_xenium_default_merged"

### LandscapeFiles Generation and Visualization for Merged Segmentation (Cellpose2 + Xenium)

In [ ]:
dega.pre.add_custom_segmentation(path_landscape_files=path_landscape_files, 
                                 path_segmentation_files=path_segmentation_files)

In [ ]:
server_address = dega.viz.get_local_server()

landscape_ist = dega.viz.Landscape(
    technology='Xenium',
    base_url = f"http://localhost:{server_address}/{path_landscape_files}",
    segmentation = 'cellpose2_Xenium_merged'
)

landscape_ist